<a href="https://colab.research.google.com/github/arauch6363-crypto/pt/blob/main/PT_github_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 - Install dependencies
!pip install selenium
!pip install unidecode
!pip install fastparquet
!pip install pyarrow
!pip install beautifulsoup4
!pip install pytz

In [ ]:
# Cell 2 - Set mode
mode = 'get_past_results'
#mode = 'get_todays_races'
#mode = 'get_both'

In [ ]:
# Cell 3 - Imports
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from datetime import date, datetime, timedelta
from selenium.webdriver.common.by import By
import pytz
from bs4 import BeautifulSoup
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
import json
import re
import pandas as pd
from unidecode import unidecode
import numpy as np
import sys
import fastparquet
import undetected_chromedriver as uc
from playwright.sync_api import sync_playwright
import json
from playwright.sync_api import sync_playwright
import json
import nest_asyncio
nest_asyncio.apply()

from playwright._impl._api_types import Error as PlaywrightError
import asyncio
from playwright.async_api import async_playwright

In [ ]:
# Cell 4 - Helper functions

async def get_web_content_async(url):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        try:
            await page.goto(url, wait_until='networkidle', timeout=60000)
            content = await page.inner_text('#__NEXT_DATA__')
            props = json.loads(content)
            return props
        finally:
            await browser.close()

def get_web_content(url, driver=None):
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(get_web_content_async(url))

def web_driver():
    return None  # not used with Playwright

def get_operator_data_odds(data, operators_priority=['PMU', 'PMU.fr', 'genybet']):
    for operator in operators_priority:
        for race in data:
            if race.get('operator') == operator:
                return race.get('runners', {})
    return None

def get_operator_data_dividends(data, operators_priority=['PMU', 'PMU.fr', 'genybet']):
    for operator in operators_priority:
        for race in data:
            if race.get('operator') == operator:
                return race.get('betDividends', {})
    return None

def generate_top_5_table(df, group_column):
    df_wins = df[df['ranking'] == 1]
    total_runs = df.groupby(group_column).size()
    win_count = df_wins.groupby(group_column).size()
    result = pd.DataFrame({
        'wins': win_count,
        'runs': total_runs
    }).fillna(0)
    result['win_percentage'] = (result['wins'] / result['runs']) * 100
    top_5 = result.sort_values(by='wins', ascending=False).head(20)
    top_5['formatted'] = top_5.apply(
        lambda row: f"{row.name} {int(row['runs'])}/{int(row['wins'])} {row['win_percentage']:.0f}%", axis=1
    )
    return top_5['formatted']

In [ ]:
# Cell 5 - repeatable_cells function (get past results)
def repeatable_cells():
    driver = None  # not needed with Playwright

    try:
        races_path = "./races.parquet"
        tracker_path = "./reload_tracker.csv"

        races = pd.read_parquet(races_path)

        date_str = races.date.max()
        date_obj = datetime.strptime(date_str, '%Y-%m-%d')
        next_day = date_obj + timedelta(days=1)
        start_date = next_day.strftime('%Y-%m-%d')

        yesterday = datetime.today() - timedelta(days=1)
        yesterday_str = yesterday.strftime('%Y-%m-%d')

        try:
            reload_tracker_df = pd.read_csv(tracker_path, dtype={'reload_started': str, 'reload_finished': str})
        except FileNotFoundError:
            reload_tracker_df = pd.DataFrame(columns=['date', 'reload_done', 'reload_started', 'reload_finished', 'races_pushed'])

        def generate_date_list(start_date_str, end_date_str):
            start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
            end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
            date_list = []
            current_date = start_date
            while current_date <= end_date:
                date_list.append(current_date.strftime("%Y-%m-%d"))
                current_date += timedelta(days=1)
            return date_list

        missing_dates = generate_date_list(start_date, yesterday_str)

        for date in missing_dates:
            if date not in reload_tracker_df['date'].values:
                new_row = {
                    'date': date,
                    'reload_done': False,
                    'reload_started': '9999-12-31',
                    'reload_finished': '9999-12-31',
                    'races_pushed': np.nan
                }
                reload_tracker_df = pd.concat([reload_tracker_df, pd.DataFrame([new_row])], ignore_index=True)

        reload_tracker_df.to_csv(tracker_path, index=False)

        pending = reload_tracker_df[reload_tracker_df['reload_done'] == False]

        if pending.empty:
            print("reload complete")
            return False

        loadDate = reload_tracker_df[reload_tracker_df['reload_done'] == False]['date'].min()
        url = 'https://www.paris-turf.com/programme-courses/' + loadDate

        my_tz = pytz.timezone('Europe/Berlin')
        timestamp_start = datetime.now(my_tz).strftime('%Y-%m-%d %H:%M:%S')

        reload_tracker_df.loc[reload_tracker_df['date'] == loadDate, 'reload_started'] = timestamp_start

        print('loaddate:', loadDate, 'progress:', round((reload_tracker_df[reload_tracker_df['reload_done']==True].shape[0]/reload_tracker_df.shape[0])*100,2),'%')
        print("Running the repeatable code")

        props = get_web_content(url)

        races = props['props']['pageProps']['initialState']['raceCardsState']['races'][loadDate]
        meetings = props['props']['pageProps']['initialState']['raceCardsState']['meetings'][loadDate]

        df_races = pd.DataFrame(races)
        df_meetings = pd.DataFrame(meetings)

        merged_df = pd.merge(df_meetings[['id', 'name', 'country']], df_races, left_on='id', right_on='meetingId', suffixes=('_meeting', '_race'))

        filtered_df = merged_df[(merged_df['country'] == 'FR') & (merged_df['discipline'] != 'Trot') & (merged_df['specialty'] == 'P') & (merged_df['breed'].isnull() == True)]

        required_columns = ['date', 'name_meeting', 'meetingId', 'trackCode', 'name_race', 'id_race', 'specialty', 'class', 'type', 'sex',
                            'surface', 'going', 'penetrometer', 'number', 'distance', 'isPremium', 'breed', 'totalPrize',
                            'uuid', 'direction', 'rail', 'winningPost', 'minAge', 'maxAge', 'discipline', 'winnerTimeKm', 'winnerTime']

        final_df = filtered_df.reindex(columns=required_columns, fill_value=np.nan)

        def clean_name(name):
            name = unidecode(name.lower())
            name = name.replace(' ', '-').replace('(', '').replace(')', '')
            return name

        final_df['cleaned_name_race'] = final_df['name_race'].apply(clean_name)
        final_df['cleaned_name_meeting'] = final_df['name_meeting'].apply(clean_name)
        final_df['race_url'] = 'https://www.paris-turf.com/course/' + final_df['cleaned_name_meeting'] + '-' + final_df['cleaned_name_race'] + '-idc-' + final_df['uuid']
        final_df.drop(columns=['cleaned_name_race', 'cleaned_name_meeting'], inplace=True)
        race_df = final_df.reset_index(drop=True)
        race_df['runners_loaded'] = False

        df_races_old = pd.read_parquet(races_path)
        df_combined = pd.concat([df_races_old, race_df])
        df_combined.to_parquet(races_path, engine='pyarrow', index=None)
        df_races_old = pd.read_parquet(races_path)

        runners_dfs = []
        webTips_dfs = []
        odds_dfs = []
        dividends_dfs = []

        for i in range(len(race_df['race_url'])):
            props = get_web_content(race_df['race_url'][i])

            runners = props['props']['pageProps']['initialState']['raceCardsState']['runners'][str(race_df['id_race'][i])]
            df_runners = pd.DataFrame(runners)

            try:
                webTips = props['props']['pageProps']['initialState'].get('currentPageState', {}).get('webTips', [])
                df_webTips = pd.DataFrame(webTips).reset_index()
            except:
                webTips = pd.DataFrame()

            try:
                odds = get_operator_data_odds(props['props']['pageProps']['initialState'].get('currentPageState', {}).get('betinRaceOdd', {}).get('odds', {}))
                df_odds = pd.DataFrame(odds).reset_index()
            except:
                df_odds = pd.DataFrame()

            try:
                dividends = get_operator_data_dividends(props['props']['pageProps']['initialState'].get('currentPageState', {}).get('dividends', {}).get('raceDividends', {}))
                df_dividends = pd.DataFrame(dividends).reset_index()
            except:
                df_dividends = pd.DataFrame()

            keys = ['horseId', 'isRunnerState', 'meetingId', 'age', 'hood', 'breederName', 'shoeingFront', 'isEngaged', 'jockeyName', 'draw', 'saddle', 'isSupplemented', 'isRunning', 'weightKg', 'raceDirection', 'numberOfPlaces', 'ownerName', 'raceId', 'uuid', 'raceSpeciality', 'noShoesFirstTime',
                    'raceTotalPrize', 'ranking', 'jockeyUUID', 'totalPrize', 'horseName', 'horseSir', 'trainerUUID', 'meetingName', 'horseUUID', 'ownerUUID', 'trainerName', 'protectionFirstTime', 'raceName', 'jockeyAllowance', 'tongueTie', 'raceIsTQQ', 'sex', 'shoeingBack',
                    'totalWinningPrize','breederId', 'margin', 'jockeyChanged', 'coloursPng', 'shoeing', 'bestImpression', 'comment', 'isPremium', 'blinkers', 'jockeyId', 'raceType', 'ownerId', 'handicapRatingKg', 'horseDam', 'weightChanged', 'claimRating',
                    'trainerId', 'blinkersFirstTime','raceNumber']

            keys2 = ['meetingId', 'raceId', 'text', 'tips']
            keys_odds = ['horseId', 'horseNumber', 'liveOdd', 'referenceOdd', 'isFavorite', 'runnerId', 'liveOddDateTime', 'referenceOddDateTime', 'runnerStatus', 'runnerSlug', 'horseName']
            keys_dividends = ['betType', 'turnover', 'dividend', 'originDividendType', 'minimumBet', 'dividendType', 'originBetType', 'combination']

            if not df_runners.empty:
                df_runners_required_columns = df_runners.reindex(columns=keys, fill_value=np.nan)

            if not df_webTips.empty:
                df_webTips_required_columns = df_webTips.reindex(columns=keys2, fill_value=np.nan)

            if not df_odds.empty:
                df_odds_required_columns = df_odds.reindex(columns=keys_odds, fill_value=np.nan)
                df_odds_required_columns['meetingId'] = race_df['meetingId'][i]
                df_odds_required_columns['raceId'] = race_df['id_race'][i]

            if not df_dividends.empty:
                df_dividends_required_columns = df_dividends.reindex(columns=keys_dividends, fill_value=np.nan)
                df_dividends_required_columns['meetingId'] = race_df['meetingId'][i]
                df_dividends_required_columns['raceId'] = race_df['id_race'][i]

            runners_dfs.append(df_runners_required_columns)

            if not df_webTips.empty:
                webTips_dfs.append(df_webTips_required_columns)

            if not df_odds.empty:
                odds_dfs.append(df_odds_required_columns)

            if not df_dividends.empty:
                dividends_dfs.append(df_dividends_required_columns)

            print(race_df['race_url'][i], df_runners_required_columns.shape)

            if df_runners_required_columns.shape[0] < 2:
                sys.exit("Notebook Execution Stopped")

        if len(runners_dfs) != 0:
            df_runners_new = pd.concat(runners_dfs, ignore_index=True)
            df_runners_old = pd.read_parquet("./runners.parquet")
            df_combined = pd.concat([df_runners_new, df_runners_old], ignore_index=True)
            df_combined.to_parquet("./runners.parquet", engine='fastparquet', index=None)
            print('old:', df_runners_old.shape[0],'+ new:', df_runners_new.shape[0],'= ', df_combined.shape[0])

            top_5_jockeys = generate_top_5_table(df_combined, 'jockeyName')
            top_5_trainers = generate_top_5_table(df_combined, 'trainerName')
            top_5_owners = generate_top_5_table(df_combined, 'ownerName')
            top_5_breeders = generate_top_5_table(df_combined, 'breederName')
            top_5_horses = generate_top_5_table(df_combined, 'horseName')
            top_5_sires = generate_top_5_table(df_combined, 'horseSir')

            print("Top 20 Jockeys:")
            print(top_5_jockeys.to_string(index=False))
            print("\nTop 20 Trainers:")
            print(top_5_trainers.to_string(index=False))
            print("\nTop 20 Owners:")
            print(top_5_owners.to_string(index=False))
            print("\nTop 20 Breeders:")
            print(top_5_breeders.to_string(index=False))
            print("\nTop 20 Horses:")
            print(top_5_horses.to_string(index=False))
            print("\nTop 20 Sires:")
            print(top_5_sires.to_string(index=False))

        if len(webTips_dfs) != 0:
            df_webTips_new = pd.concat(webTips_dfs, ignore_index=True)
            df_webTips_old = pd.read_parquet("./webTips.parquet")
            df_combined = pd.concat([df_webTips_new, df_webTips_old], ignore_index=True)
            df_combined.to_parquet("./webTips.parquet", engine='pyarrow', index=None)

        if len(odds_dfs) != 0:
            df_odds_new = pd.concat(odds_dfs, ignore_index=True)
            df_odds_old = pd.read_parquet("./odds.parquet")
            df_combined = pd.concat([df_odds_new, df_odds_old], ignore_index=True)
            df_combined.to_parquet("./odds.parquet", engine='pyarrow', index=None)

        if len(dividends_dfs) != 0:
            df_dividends_new = pd.concat(dividends_dfs, ignore_index=True)
            df_dividends_old = pd.read_parquet("./dividends.parquet")
            df_combined = pd.concat([df_dividends_new, df_dividends_old], ignore_index=True)
            df_combined.to_parquet("./dividends.parquet", engine='pyarrow', index=None)

        timestamp_finished = datetime.now(my_tz).strftime('%Y-%m-%d %H:%M:%S')
        races_cnt = race_df['id_race'].nunique()
        reload_tracker_df.loc[reload_tracker_df['date'] == loadDate, 'reload_done'] = True
        reload_tracker_df.loc[reload_tracker_df['date'] == loadDate, 'reload_finished'] = timestamp_finished
        reload_tracker_df.loc[reload_tracker_df['date'] == loadDate, 'races_pushed'] = races_cnt
        reload_tracker_df.to_csv(tracker_path, index=False)

        return True
    finally:
        pass  # driver.quit() not needed with Playwright

In [ ]:
# Cell 7 - Run past results
if mode in ('get_past_results', 'get_both'):
    for i in range(10):
        print(f"Iteration {i+1}")
        if not repeatable_cells():
            break